In [44]:
import numpy as np
import pandas as pd
from pathlib import Path

In [45]:
rng = np.random.default_rng(42)
N_sites = 100
countries = ["Germany", "France", "Italy", "Spain", "Portugal"]
indications = [
"Oncology", "Neurology", "Rheumatology", "Psychiatry",
"Cardiology", "Dermatology", "Endocrinology",
"Immunology", "Infectious Diseases"
]

In [46]:
#site characteristics

df_sites = pd.DataFrame({
"site_id": range(1, N_sites + 1),
"country": rng.choice(countries, size=N_sites),
"indication": rng.choice(indications, size=N_sites),
"eligible_patients_monthly": rng.integers(5, 35, size=N_sites),
"screen_failure_rate": rng.uniform(0.1, 0.5, size=N_sites).round(2),
"competing_studies": rng.integers(0, 3, size=N_sites),
"activation_days": rng.integers(30, 180, size=N_sites),
"staff_capacity": rng.integers(1, 11, size=N_sites),
"data_quality_score": rng.integers(50, 101, size=N_sites),
"recruitment_rate": rng.uniform(0.1, 0.5, size=N_sites).round(2)
})
display(df_sites.head())
print("Number of unique sites:", df_sites["site_id"].nunique())


,site_id,country,indication,eligible_patients_monthly,screen_failure_rate,competing_studies,activation_days,staff_capacity,data_quality_score,recruitment_rate
0,1,Germany,Immunology,15,0.43,0,122,6,50,0.12
1,2,Spain,Neurology,32,0.46,1,146,7,65,0.45
2,3,Spain,Immunology,19,0.16,2,140,2,76,0.38
3,4,Italy,Oncology,25,0.32,2,50,2,96,0.17
4,5,Italy,Immunology,18,0.14,0,153,2,87,0.14


Number of unique sites: 100


In [47]:
#competition adjustment
df_sites["competition_factor"] = (
1 / (1 + 0.15 * df_sites["competing_studies"])
).round(3)

In [48]:
#adjustment potential
df_sites["adjusted_recruitment_monthly"] = (
df_sites["eligible_patients_monthly"]
* (1 - df_sites["screen_failure_rate"])
* df_sites["recruitment_rate"]
* df_sites["competition_factor"]
).round(2)

In [ ]:
#multiple indications per site
site_indications = []

for site_id in df_sites["site_id"]:
    n_indications = rng.integers(2, 6)
    selected = rng.choice(
        indications,
        size=n_indications,
        replace=False
    )

    for indication in selected:
        site_indications.append({
            "site_id": site_id,
            "indication": indication
        })
df_sites = pd.DataFrame(site_indications)

print("Total site-indication combinations:", len(df_indications))
print("Unique sites:", df_indications["site_id"].nunique())
print("Unique indications:", df_indications["indication"].nunique())

display(df_indications.head())

Total site-indication combinations: 356
Unique sites: 100
Unique indications: 9


,site_id,indication
0,1,Immunology
1,1,Neurology
2,1,Psychiatry
3,2,Infectious Diseases
4,2,Immunology


In [50]:
#indications
df_indications = pd.DataFrame(site_indications)

n_rows = len(df_indications)

df_indications["eligible_patients_monthly"] = rng.integers(
    5, 35, size=n_rows
)

df_indications["screen_failure_rate"] = rng.uniform(
    0.1, 0.5, size=n_rows
).round(2)

df_indications["competing_studies"] = rng.integers(
    0, 6, size=n_rows
)

df_indications["recruitment_rate"] = rng.uniform(
    0.1, 0.5, size=n_rows
).round(2)

df_indications["competition_factor"] = (
    1 / (1 + 0.15 * df_indications["competing_studies"])
)

df_indications["adjusted_recruitment_monthly"] = (
    df_indications["eligible_patients_monthly"]
    * (1 - df_indications["screen_failure_rate"])
    * df_indications["recruitment_rate"]
    * df_indications["competition_factor"]
).round(2)

In [51]:
print(
    df_indications[
        ["site_id", "indication", "eligible_patients_monthly"]
    ]
    .sort_values("site_id")
    .head(15)
)

    site_id           indication  eligible_patients_monthly
0         1           Immunology                         25
1         1            Neurology                         19
2         1           Psychiatry                         31
3         2  Infectious Diseases                          8
4         2           Immunology                         12
5         3            Neurology                         12
6         3  Infectious Diseases                          9
7         3         Rheumatology                         13
8         3        Endocrinology                         14
9         4  Infectious Diseases                         27
10        4             Oncology                         14
11        4           Immunology                         31
12        4         Rheumatology                         28
13        5           Cardiology                         32
14        5           Psychiatry                          8


In [52]:
#combine indication-specific and site-level data
df_sites = df_indications.merge(
    df_sites,
    on="site_id",
    how="left"
)

In [54]:
#illustrative operational risk
df_sites["risk_level"] = np.select(
    [
        (df_sites["activation_days"] > 120)
        | (df_sites["data_quality_score"] < 65)
        | (df_sites["staff_capacity"] <= 3)
        | (df_sites["competing_studies"] >= 3),

        (df_sites["activation_days"] > 90)
        | (df_sites["data_quality_score"] < 80)
        | (df_sites["staff_capacity"] <= 5)
        | (df_sites["competing_studies"] >= 2),
    ],
    ["High", "Medium"],
    default="Low"
)

KeyError: 'competing_studies'

In [ ]:
#one row per site-indication combination
df_indications = df_indications.merge(
    df_sites[
        [
            "site_id",
            "country",
            "activation_days",
            "staff_capacity",
            "data_quality_score"
        ]
    ],
    on="site_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
#output
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

df_indications.to_csv(output_dir / "sites_data.csv", index=False)

print(f"Generated {len(df_indications)} site-indication combinations")
print(f"Unique sites: {df_indications['site_id'].nunique()}")
print(f"Data saved to {output_dir / 'sites_data.csv'}")
print(df_indications.head())

Generated 353 site-indication combinations
Unique sites: 100
Data saved to data\sites_data.csv
   site_id     indication  eligible_patients_monthly  screen_failure_rate  \
0        1      Neurology                         12                 0.23   
1        1     Immunology                         32                 0.12   
2        1     Cardiology                         33                 0.14   
3        1     Psychiatry                          6                 0.25   
4        1  Endocrinology                         23                 0.12   

   competing_studies  recruitment_rate  competition_factor  \
0                  3              0.50            0.689655   
1                  1              0.12            0.869565   
2                  5              0.26            0.571429   
3                  5              0.23            0.571429   
4                  5              0.48            0.571429   

   adjusted_recruitment_monthly country_x  activation_days_x  \
0    

In [ ]:
df_sites.shape
print(df_indications.groupby("site_id")["indication"].nunique().value_counts().sort_index())
print(df_indications["indication"].value_counts())
print(df_indications["indication"].nunique())

indication
2    25
3    24
4    24
5    27
Name: count, dtype: int64
indication
Cardiology             47
Endocrinology          42
Neurology              40
Dermatology            40
Immunology             39
Oncology               38
Psychiatry             37
Infectious Diseases    37
Rheumatology           33
Name: count, dtype: int64
9


In [ ]:
print(df_sites.shape)
print(df_sites.columns.tolist())
print(df_sites["site_id"].nunique())

(100, 5)
['site_id', 'country', 'activation_days', 'staff_capacity', 'data_quality_score']
100


In [ ]:
df_check = pd.read_csv("data/sites_data.csv")

print(
    df_check[
        ["site_id", "indication", "eligible_patients_monthly"]
    ]
    .sort_values("site_id")
    .head(15)
)

    site_id           indication  eligible_patients_monthly
0         1            Neurology                         12
1         1           Immunology                         32
2         1           Cardiology                         33
3         1           Psychiatry                          6
4         1        Endocrinology                         23
5         2         Rheumatology                         11
6         2            Neurology                         10
7         3  Infectious Diseases                         13
8         3          Dermatology                         10
9         4           Psychiatry                         19
10        4           Immunology                         34
11        4        Endocrinology                         22
12        5           Immunology                         26
13        5        Endocrinology                         19
14        5            Neurology                          5
